In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import shlex
import subprocess

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend
from deepagents.backends import LocalShellBackend

from deepagents.backends.protocol import ExecuteResponse, SandboxBackendProtocol
from langchain.tools import ToolRuntime
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model


import sys
from pathlib import Path

ROOT = Path("/Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.agents.drug_target.tools import (
    sequential_thinking,
)

In [ ]:
dir = "tests/drugtarget"
HOST_OUTPUT_DIR = os.path.abspath(f"{ROOT}/{dir}")
print(HOST_OUTPUT_DIR)

In [ ]:
import os
import docker
from deepagents import create_deep_agent
from deepagents.backends.sandbox import BaseSandbox
from deepagents.backends.protocol import ExecuteResponse, FileDownloadResponse, FileUploadResponse

# 1. 호스트(내 PC)에 결과물을 저장할 폴더 생성
HOST_OUTPUT_DIR = os.path.abspath(f"{ROOT}/{dir}")
os.makedirs(HOST_OUTPUT_DIR, exist_ok=True)

# 2. BaseSandbox를 상속하여 execute()만 구현하면 grep_raw, read, write 등 자동 제공
class DockerBackend(BaseSandbox):
    def __init__(self, image_name="python:3.11-slim", host_dir=None, container_dir="/workspace"):
        self.client = docker.from_env()
        self.container_dir = container_dir
        self._image_name = image_name
        
        # 볼륨 설정: 호스트 폴더를 컨테이너의 /workspace 에 읽기/쓰기(rw) 모드로 연결
        volumes_config = {}
        if host_dir:
            print(f"🔗 폴더 마운트: {host_dir} -> {container_dir}")
            volumes_config = {
                host_dir: {'bind': container_dir, 'mode': 'rw'}
            }

        print(f"📦 Docker 컨테이너({image_name})를 백엔드로 시작합니다...")
        
        self.container = self.client.containers.run(
            image_name, 
            command="tail -f /dev/null", 
            detach=True,
            remove=True,
            volumes=volumes_config,
            working_dir=container_dir
        )

    @property
    def id(self) -> str:
        return self.container.short_id

    def execute(self, command: str, *, timeout: int | None = None) -> ExecuteResponse:
        """에이전트가 셸 명령어를 요청하면 Docker 컨테이너 내부에서 실행"""
        #print(f"\n[{self.container.short_id[:8]} 실행] {command[:200]}")
        
        exit_code, output = self.container.exec_run(["/bin/sh", "-c", command])
        decoded_output = output.decode('utf-8')
        return ExecuteResponse(output=decoded_output, exit_code=exit_code)

    def upload_files(self, files: list[tuple[str, bytes]]) -> list[FileUploadResponse]:
        results = []
        for path, content in files:
            try:
                import tarfile, io
                tar_stream = io.BytesIO()
                with tarfile.open(fileobj=tar_stream, mode='w') as tar:
                    info = tarfile.TarInfo(name=os.path.basename(path))
                    info.size = len(content)
                    tar.addfile(info, io.BytesIO(content))
                tar_stream.seek(0)
                self.container.put_archive(os.path.dirname(path) or "/", tar_stream)
                results.append(FileUploadResponse(path=path))
            except Exception as e:
                results.append(FileUploadResponse(path=path, error="permission_denied"))
        return results

    def download_files(self, paths: list[str]) -> list[FileDownloadResponse]:
        results = []
        for path in paths:
            try:
                import tarfile, io
                bits, _ = self.container.get_archive(path)
                stream = io.BytesIO(b"".join(bits))
                with tarfile.open(fileobj=stream) as tar:
                    member = tar.getmembers()[0]
                    content = tar.extractfile(member).read()
                results.append(FileDownloadResponse(path=path, content=content))
            except Exception:
                results.append(FileDownloadResponse(path=path, error="file_not_found"))
        return results

    def cleanup(self):
        """작업 완료 후 컨테이너 종료"""
        print("🧹 Docker 컨테이너를 종료 및 정리합니다...")
        self.container.stop()


# 3. 마운트 설정이 포함된 Docker 백엔드 인스턴스화
my_docker_backend = DockerBackend(
    image_name="python:3.11-slim",
    host_dir=HOST_OUTPUT_DIR,
    container_dir="/workspace"
)

# 4. 에이전트 설정
research_subagent = {
    "name": "researcher",
    "description": "Research assistant",
    "system_prompt": "You are a researcher. You must save all your output files and code in the '/workspace' directory.",
    "skills": ["/skills/kegg-database", "skills/compare-census-gene-pair-by-cell-type", "skills/cellxgene-census"],
}

agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    subagents=[research_subagent],
    backend=my_docker_backend
)

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    #"brca1 과 관련잇는 pathway는 무엇인가요"
                    "TGFB1와 TGFBR2가 breast cancer 에서 invasive tumor cell에서 다른 세포타입에 비해 유의미하게 발현되는지"
                ),
            }
        ]
    }
)

print(result["messages"][-1].content)

In [ ]:
# 작업이 끝나거나 에러가 나도 컨테이너를 안전하게 정리
my_docker_backend.cleanup()
print(f"✅ 작업 완료! 결과물은 호스트의 {HOST_OUTPUT_DIR} 폴더를 확인하세요.")

from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langchain.chat_models import init_chat_model

# 로컬 환경에서 셸 명령어와 파일 시스템에 직접 접근할 수 있도록 백엔드 설정
# root_dir을 프로젝트 경로에 맞게 지정하세요.
local_backend = LocalShellBackend(
    root_dir=".", 
    env={"PATH": "/usr/bin:/bin"} # 파이썬 실행이 가능하도록 환경 변수 설정
)

research_subagent = {
    "name": "researcher",
    "description": "Research assistant with specialized skills",
    "system_prompt": "You are a researcher.",
    "tools": [sequential_thinking],
    "skills": ["/skills/kegg-database/", "/skills/string-database/"],
}

agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    skills=["/skills/main/"],
    subagents=[research_subagent],
    backend=local_backend  # 샌드박스 대신 로컬 셸 백엔드 주입
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "brca1 과 관련잇는 pathway는 무엇인가요"
                ),
            }
        ]
    }
)

print(result["messages"][-1].content)
